# Bài 5.2 · CNN và ResNet trên ảnh thật

**Câu hỏi của buổi này:** một CNN nhìn ảnh qua từng tầng ra sao, và vì sao mạng rất sâu cần đường đi tắt mới học được?

Dữ liệu là **CIFAR-10**: 60 000 ảnh màu 32×32 của 10 lớp (máy bay, ô tô, chim, mèo…). Ta tự huấn luyện một CNN nhỏ, nhìn vào bên trong nó, rồi làm thí nghiệm đường tắt với mạng 34 tầng.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập, ảnh mẫu 10 lớp |
| 2. Huấn luyện một CNN nhỏ | bảng cấu trúc, đường loss và accuracy |
| 3. Kết quả từng lớp | accuracy từng lớp, confusion matrix, ảnh đoán sai |
| 4. Bên trong CNN | bộ lọc tầng đầu của ResNet-18, feature map qua ba block |
| 5. Đường tắt | gradient từng tầng và đường loss của ba mạng sâu |
| 6. ResNet-18 và ResNet-50 | bảng shape và số tham số từng giai đoạn |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 5 phút. Bài giảng: [ai-canuc80k · bài 5.2](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.2).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, random
import torch, torchvision
from torch import nn
import torch.nn.functional as F
import datasets
from datasets import load_dataset
datasets.disable_progress_bars()

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

seed_all(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TRAIN, N_TEST, EPOCHS, DEEP_STEPS = 50000, 10000, 12, 400
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Tải CIFAR-10 từ Hugging Face, đổi cả tập thành một tensor để huấn luyện nhanh trên GPU. Mỗi kênh màu được chuẩn hóa bằng mean và std **tính trên train**.

In [ ]:
with quiet():
    cifar = load_dataset('uoft-cs/cifar10')
VI = {'airplane': 'máy bay', 'automobile': 'ô tô', 'bird': 'chim', 'cat': 'mèo', 'deer': 'hươu',
      'dog': 'chó', 'frog': 'ếch', 'horse': 'ngựa', 'ship': 'tàu thủy', 'truck': 'xe tải'}
vi = [VI[c] for c in cifar['train'].features['label'].names]

def to_tensor(split, n):
    d = split.select(range(min(n, len(split))))
    x = torch.from_numpy(np.stack([np.asarray(im.convert('RGB')) for im in d['img']])).permute(0, 3, 1, 2).contiguous()
    return x, torch.tensor(d['label'])

x_tr8, y_tr = to_tensor(cifar['train'], N_TRAIN)
x_te8, y_te = to_tensor(cifar['test'], N_TEST)
mean = x_tr8.float().div(255).mean((0, 2, 3)); std = x_tr8.float().div(255).std((0, 2, 3))
norm = lambda x8: ((x8.float().div(255) - mean[:, None, None]) / std[:, None, None])
X_tr, X_te = norm(x_tr8).to(device), norm(x_te8).to(device)
Y_tr, Y_te = y_tr.to(device), y_te.to(device)

table(pd.DataFrame([('train', len(y_tr), tuple(X_tr.shape)), ('test', len(y_te), tuple(X_te.shape))], columns=['tập', 'số ảnh', 'tensor']), 'Hai tập')
table(pd.DataFrame({'kênh': ['R', 'G', 'B'], 'mean (train)': mean.numpy(), 'std (train)': std.numpy()}), 'Thống kê dùng để chuẩn hóa', fmt={'mean (train)': '{:.3f}', 'std (train)': '{:.3f}'})
check('sau chuẩn hóa, train có mean mỗi kênh gần 0', X_tr.mean((0, 2, 3)).abs().max().item() < 1e-3)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for k in range(10):
    ids = torch.nonzero(y_tr == k).flatten()[:2]
    for r in range(2):
        axes[r, k].imshow(x_tr8[ids[r]].permute(1, 2, 0)); axes[r, k].axis('off')
    axes[0, k].set_title(vi[k], fontsize=10)
fig.suptitle('Mỗi lớp hai ảnh: chỉ 32×32 pixel', y=1.02)
plt.tight_layout(); plt.show()

## 2. Huấn luyện một CNN nhỏ

Ba block, mỗi block gồm hai lớp conv 3×3 (kèm BatchNorm, ReLU) rồi max-pool 2×2: ảnh co từ 32 → 16 → 8 → 4 pixel, số kênh tăng 32 → 64 → 128. Cuối cùng lấy trung bình mỗi kênh (global average pooling) và một lớp Linear ra 10 lớp. Khi train, mỗi ảnh có 50% bị lật ngang.

In [ ]:
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1, self.b2, self.b3 = block(3, 32), block(32, 64), block(64, 128)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10))
    def forward(self, x):
        return self.head(self.b3(self.b2(self.b1(x))))

seed_all(0)
cnn = SmallCNN().to(device)
x = torch.zeros(1, 3, 32, 32, device=device); rows = []
for name in ['b1', 'b2', 'b3', 'head']:
    mod = getattr(cnn, name); x = mod(x)
    rows.append((name, tuple(x.shape[1:]), sum(p.numel() for p in mod.parameters())))
table(pd.DataFrame(rows, columns=['phần', 'shape đầu ra', 'số tham số']), f'Cấu trúc CNN nhỏ ({sum(r[2] for r in rows):,} tham số)', fmt={'số tham số': '{:,}'})

In [ ]:
def evaluate(model, X, Y, bs=1000):
    model.eval(); loss, ok = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(Y), bs):
            out = model(X[i:i + bs]); loss += F.cross_entropy(out, Y[i:i + bs], reduction='sum').item(); ok += (out.argmax(1) == Y[i:i + bs]).sum().item()
    return loss / len(Y), ok / len(Y)

def batches(n, bs, gen):
    perm = torch.randperm(n, generator=gen).to(device)
    for i in range(0, n, bs):
        yield perm[i:i + bs]

BS = 256
opt = torch.optim.AdamW(cnn.parameters(), lr=3e-3, weight_decay=5e-4)
steps = EPOCHS * ((len(Y_tr) + BS - 1) // BS)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=steps)
gen, hist, t0 = torch.Generator().manual_seed(0), [], time.time()
for ep in range(1, EPOCHS + 1):
    cnn.train(); tot, ok, n = 0.0, 0, 0
    for idx in batches(len(Y_tr), BS, gen):
        xb, yb = X_tr[idx], Y_tr[idx]
        flip = torch.rand(len(idx), device=device) < 0.5
        xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
        out = cnn(xb); loss = F.cross_entropy(out, yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(idx); ok += (out.argmax(1) == yb).sum().item(); n += len(idx)
    hist.append((ep, tot / n, ok / n, *evaluate(cnn, X_te, Y_te)))
H = pd.DataFrame(hist, columns=['epoch', 'train loss', 'train acc', 'test loss', 'test acc'])
table(H, f'{EPOCHS} epoch trên {len(Y_tr)} ảnh ({time.time() - t0:.0f} giây)', fmt={c: '{:.4f}' for c in H.columns[1:]}, best='test acc')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for col, c, lab in [('train', C['blue'], 'train'), ('test', C['clay'], 'test')]:
    axes[0].plot(H['epoch'], H[f'{col} loss'], marker='o', ms=3, color=c, label=lab)
    axes[1].plot(H['epoch'], H[f'{col} acc'], marker='o', ms=3, color=c, label=lab)
axes[0].set(xlabel='epoch', ylabel='loss', title='Loss'); axes[1].set(xlabel='epoch', ylabel='accuracy', title='Accuracy')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()
test_acc = H['test acc'].iloc[-1]
check(f'test accuracy {test_acc:.3f} vượt xa đoán bừa 0.1', test_acc > 0.5)

## 3. Kết quả từng lớp

In [ ]:
cnn.eval()
with torch.no_grad():
    prob = torch.cat([cnn(X_te[i:i + 1000]).softmax(1) for i in range(0, len(Y_te), 1000)]).cpu()
pred = prob.argmax(1)
per = pd.DataFrame({'lớp': vi, 'accuracy': [(pred[y_te == k] == k).float().mean().item() for k in range(10)]})

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.15]})
order = per.sort_values('accuracy')
axes[0].barh(order['lớp'], order['accuracy'], color=[C['clay'] if v == order['accuracy'].min() else C['blue'] for v in order['accuracy']])
axes[0].set_xlim(0, 1.1); axes[0].set_title('Accuracy từng lớp (test)'); axes[0].grid(axis='y', visible=False); bar_labels(axes[0], '{:.2f}')
cm = np.zeros((10, 10), int)
for a, b in zip(y_te.tolist(), pred.tolist()): cm[a, b] += 1
axes[1].imshow(cm, cmap='Blues'); axes[1].grid(False)
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] >= cm.sum() * 0.004:
            axes[1].text(j, i, cm[i, j], ha='center', va='center', fontsize=8, color=C['clay'])
axes[1].set_xticks(range(10), vi, rotation=45, ha='right'); axes[1].set_yticks(range(10), vi)
axes[1].set(xlabel='dự đoán', ylabel='nhãn thật', title='Confusion matrix (số đỏ: nhầm nhiều)')
plt.tight_layout(); plt.show()

off = cm - np.diag(np.diag(cm)); a, b = np.unravel_index(off.argmax(), off.shape)
note(f"Lớp khó nhất: <b>{order['lớp'].iloc[0]}</b> ({order['accuracy'].iloc[0]:.2f}); dễ nhất: {order['lớp'].iloc[-1]} ({order['accuracy'].iloc[-1]:.2f}). "
     f"Cặp nhầm nhiều nhất: <b>{vi[a]}</b> bị đoán thành <b>{vi[b]}</b> ({off[a, b]} ảnh).")

wrong = torch.nonzero((pred != y_te) & (y_te == a)).flatten()[:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(1.9 * len(wrong), 2.4))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(x_te8[i].permute(1, 2, 0)); ax.axis('off')
    ax.set_title(f'đoán: {vi[pred[i]]}\n({prob[i].max():.2f})', fontsize=9, color=C['clay'])
fig.suptitle(f'Ảnh {vi[a]} bị đoán sai', y=1.08)
plt.tight_layout(); plt.show()

## 4. Bên trong CNN

**Bộ lọc tầng đầu.** CNN nhỏ của ta dùng bộ lọc 3×3, quá bé để nhìn. ResNet-18 học trên ImageNet có 64 bộ lọc 7×7 ở tầng đầu; mỗi ô dưới đây là một bộ lọc, vẽ như một ảnh màu nhỏ.

In [ ]:
with quiet():
    r18 = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
w = r18.conv1.weight.detach().cpu()
check(f'tầng đầu ResNet-18: {w.shape[0]} bộ lọc, mỗi bộ lọc {w.shape[1]}×{w.shape[2]}×{w.shape[3]}', tuple(w.shape) == (64, 3, 7, 7))
fig, axes = plt.subplots(4, 16, figsize=(14, 3.8))
for ax, f in zip(axes.flat, w):
    f = (f - f.min()) / (f.max() - f.min())
    ax.imshow(f.permute(1, 2, 0)); ax.axis('off')
fig.suptitle('64 bộ lọc 7×7 tầng đầu của ResNet-18: nét theo nhiều hướng, vệt màu, chấm sáng tối', y=1.02)
plt.tight_layout(); plt.show()

**Feature map.** Cho một ảnh test đi qua CNN nhỏ của ta và chụp đầu ra sau mỗi block. Mỗi ô là một kênh: chỗ sáng là chỗ bộ lọc của kênh ấy phản ứng mạnh. Càng lên cao, ảnh càng nhỏ và kênh càng khó đọc bằng mắt, vì mỗi ô nhìn một vùng rộng hơn của ảnh gốc.

In [ ]:
i = int(torch.nonzero((y_te == vi.index('ngựa')) & (pred == y_te)).flatten()[0])
maps = {}
hooks = [getattr(cnn, n).register_forward_hook(lambda m, inp, out, n=n: maps.__setitem__(n, out[0].detach().cpu())) for n in ['b1', 'b2', 'b3']]
with torch.no_grad(): cnn(X_te[i:i + 1])
for h in hooks: h.remove()

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax in axes.flat: ax.axis('off')
axes[0, 0].imshow(x_te8[i].permute(1, 2, 0)); axes[0, 0].set_title('ảnh vào\n3 kênh, 32×32', fontsize=11)
for c, n in enumerate(['b1', 'b2', 'b3'], start=1):
    fm = maps[n]; top = fm.flatten(1).std(1).argsort(descending=True)[:4]      # kênh có vùng sáng tối rõ nhất
    for r, k in enumerate(top):
        axes[r, c].imshow(fm[k], cmap='magma')
    axes[0, c].set_title(f'sau block {c}\n{fm.shape[0]} kênh, {fm.shape[1]}×{fm.shape[2]}', fontsize=11)
fig.suptitle('Mỗi cột: 4 kênh có vùng sáng tối rõ nhất sau một block', y=1.0)
plt.tight_layout(); plt.show()
check('feature map co 16 → 8 → 4 pixel', [maps[n].shape[1] for n in ['b1', 'b2', 'b3']] == [16, 8, 4])

## 5. Vì sao mạng sâu cần đường tắt

Dựng ba mạng cùng độ sâu: một conv đầu, **16 block** mỗi block hai conv 3×3 (32 kênh), rồi lớp phân loại — tổng cộng 34 tầng có trọng số. Ba mạng chỉ khác nhau ở hai điều:

| Mạng | BatchNorm | Đường tắt $y = \mathrm{ReLU}(x + F(x))$ |
|---|---|---|
| A | không | không |
| B | có | không |
| C | có | có |

Đo độ lớn gradient của từng tầng ở bước đầu tiên, rồi huấn luyện mỗi mạng cùng số bước, cùng dữ liệu.

In [ ]:
class DeepBlock(nn.Module):
    def __init__(self, c, skip, bn):
        super().__init__()
        Norm = nn.BatchNorm2d if bn else nn.Identity
        self.skip = skip
        self.f = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), Norm(c), nn.ReLU(), nn.Conv2d(c, c, 3, padding=1), Norm(c))
    def forward(self, x):
        return F.relu(x + self.f(x)) if self.skip else F.relu(self.f(x))

def deep_net(skip, bn, blocks=16):
    torch.manual_seed(0)
    return nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, stride=2), nn.ReLU(), *[DeepBlock(32, skip, bn) for _ in range(blocks)],
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10)).to(device)

configs = [('A: không BN, không tắt', False, False, C['gray']), ('B: có BN, không tắt', False, True, C['sand']), ('C: có BN, có tắt', True, True, C['blue'])]
deep_res, grads, curves = [], {}, {}
for name, skip, bn, _ in configs:
    net = deep_net(skip, bn); o = torch.optim.Adam(net.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(0); losses = []
    convs = [m for m in net.modules() if isinstance(m, nn.Conv2d)]
    for step in range(DEEP_STEPS):
        net.train(); idx = torch.randint(0, len(Y_tr), (128,), generator=gen).to(device)
        loss = F.cross_entropy(net(X_tr[idx]), Y_tr[idx]); o.zero_grad(); loss.backward()
        if step == 0: grads[name] = [c.weight.grad.norm().item() for c in convs]
        o.step(); losses.append(loss.item())
    curves[name] = losses
    deep_res.append((name, grads[name][0], np.mean(losses[-50:]), evaluate(net, X_te, Y_te)[1]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
for name, _, _, c in configs:
    axes[0].plot(range(1, len(grads[name]) + 1), grads[name], marker='o', ms=3, color=c, label=name)
    axes[1].plot(pd.Series(curves[name]).rolling(20, min_periods=1).mean(), color=c, label=name)
axes[0].set_yscale('log'); axes[0].set(xlabel='tầng conv (1 = gần ảnh nhất)', ylabel='độ lớn gradient (thang log)', title='Gradient ở bước đầu, từng tầng')
axes[1].axhline(np.log(10), color='#999', ls='--', lw=1); axes[1].text(5, np.log(10) + 0.03, 'ln 10: đoán đều 10 lớp', fontsize=9, color='#777')
axes[1].set(xlabel='bước', ylabel='loss (trung bình trượt 20 bước)', title=f'Huấn luyện {DEEP_STEPS} bước')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()
deep_df = pd.DataFrame(deep_res, columns=['mạng', 'gradient tầng 1', 'loss 50 bước cuối', 'test acc'])
table(deep_df, 'Ba mạng 34 tầng', fmt={'gradient tầng 1': '{:.2e}', 'loss 50 bước cuối': '{:.3f}', 'test acc': '{:.3f}'}, best='test acc')
ga, gc = deep_df['gradient tầng 1'].iloc[0], deep_df['gradient tầng 1'].iloc[2]
check('mạng có đường tắt học tốt hơn mạng không BN, không tắt', deep_df['test acc'].iloc[2] > deep_df['test acc'].iloc[0])
note(f"Ở tầng gần ảnh nhất, gradient của mạng A là <b>{ga:.1e}</b>, của mạng C là {gc:.1e}, lớn hơn khoảng 10<sup>{np.log10(gc / ga):.0f}</sup> lần. "
     f"Sau {DEEP_STEPS} bước, test acc lần lượt là {deep_df['test acc'].iloc[0]:.3f}, {deep_df['test acc'].iloc[1]:.3f}, <b>{deep_df['test acc'].iloc[2]:.3f}</b>.")

## 6. Từ block nhỏ đến ResNet-18 và ResNet-50

Cả hai đều có một phần đầu (conv 7×7 stride 2, max-pool), bốn giai đoạn block có đường tắt, và lớp phân loại. ResNet-18 dùng block hai conv 3×3; ResNet-50 dùng block **bottleneck** (1×1 → 3×3 → 1×1) và nhiều block hơn. Cho một ảnh 224×224 đi qua để đo shape.

In [ ]:
def stages(model):
    return [('đầu (conv, pool)', nn.Sequential(model.conv1, model.bn1, model.relu, model.maxpool)),
            ('giai đoạn 1', model.layer1), ('giai đoạn 2', model.layer2), ('giai đoạn 3', model.layer3), ('giai đoạn 4', model.layer4),
            ('pool + fc', nn.Sequential(model.avgpool, nn.Flatten(), model.fc))]

info = {}
for name, ctor in [('ResNet-18', torchvision.models.resnet18), ('ResNet-50', torchvision.models.resnet50)]:
    m = ctor(weights=None).eval(); x = torch.zeros(1, 3, 224, 224); rows = []
    with torch.no_grad():
        for sname, mod in stages(m):
            x = mod(x); nb = len(mod) if sname.startswith('giai') else ''
            rows.append((sname, nb, 'x'.join(map(str, x.shape[1:])), sum(p.numel() for p in mod.parameters())))
    info[name] = pd.DataFrame(rows, columns=['phần', 'số block', 'shape', 'tham số'])
both = info['ResNet-18'].merge(info['ResNet-50'], on='phần', suffixes=(' (18)', ' (50)'))
table(both, 'Một ảnh 224×224 đi qua hai mạng', fmt={'tham số (18)': '{:,}', 'tham số (50)': '{:,}'})

ax = both.set_index('phần')[['tham số (18)', 'tham số (50)']].div(1e6).plot.bar(figsize=(10, 3.8), color=[C['blue'], C['clay']], rot=0, width=.75)
ax.set_ylabel('triệu tham số'); ax.set_xlabel(''); ax.set_title('Tham số dồn về các giai đoạn cuối, nơi có nhiều kênh nhất'); bar_labels(ax, '{:.1f}')
plt.tight_layout(); plt.show()
tot18, tot50 = info['ResNet-18']['tham số'].sum(), info['ResNet-50']['tham số'].sum()
check('ResNet-18 khoảng 11.7 triệu tham số', abs(tot18 / 1e6 - 11.7) < 0.1)
note(f'ResNet-18: <b>{tot18 / 1e6:.1f}</b> triệu tham số; ResNet-50: <b>{tot50 / 1e6:.1f}</b> triệu. CNN nhỏ ở mục 2: {sum(p.numel() for p in cnn.parameters()) / 1e6:.2f} triệu.')

## 7. Tóm tắt

In [ ]:
summary = [('CNN nhỏ: test accuracy', f'{test_acc:.3f}'),
           ('lớp khó nhất / dễ nhất', f"{order['lớp'].iloc[0]} {order['accuracy'].iloc[0]:.2f} / {order['lớp'].iloc[-1]} {order['accuracy'].iloc[-1]:.2f}"),
           ('34 tầng, không BN, không tắt: test acc', f"{deep_df['test acc'].iloc[0]:.3f}"),
           ('34 tầng, có BN, không tắt: test acc', f"{deep_df['test acc'].iloc[1]:.3f}"),
           ('34 tầng, có BN, có tắt: test acc', f"{deep_df['test acc'].iloc[2]:.3f}"),
           ('tham số ResNet-18 / ResNet-50', f'{tot18 / 1e6:.1f} / {tot50 / 1e6:.1f} triệu')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** bỏ lật ngang ở mục 2 rồi xem khoảng cách train/test; tăng `blocks=16` lên 30 ở mục 5; thay CNN nhỏ bằng `torchvision.models.resnet18(num_classes=10)` (nhớ ảnh chỉ 32×32).